In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim



In [ ]:
corpus = [
    "I love machine learning",
    "word2vec is great algorithm",
    "Implementing word2vec is fun"
]

In [ ]:

sentences=[s.lower().split()for s in corpus]

In [ ]:
sentences

[['i', 'love', 'machine', 'learning'],
 ['word2vec', 'is', 'great', 'algorithm'],
 ['implementing', 'word2vec', 'is', 'fun']]

In [ ]:
vocab=sorted(set(word for sent in sentences for word in sent))

In [ ]:
vocab

['algorithm',
 'fun',
 'great',
 'i',
 'implementing',
 'is',
 'learning',
 'love',
 'machine',
 'word2vec']

In [ ]:
word2idx={w:i for i,w in enumerate(vocab)}
idx2word={i:w for w,i in word2idx.items()}
idx2word={i:w for i,w in enumerate(vocab)}

In [ ]:
vocab_size = len(vocab)

In [ ]:
idx2word

{0: 'algorithm',
 1: 'fun',
 2: 'great',
 3: 'i',
 4: 'implementing',
 5: 'is',
 6: 'learning',
 7: 'love',
 8: 'machine',
 9: 'word2vec'}

In [ ]:
X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])


X = torch.tensor(X)
Y = torch.tensor(Y)


In [ ]:
print(X)
print(Y)

tensor([3, 7, 8, 9, 5, 2, 4, 9, 5])
tensor([7, 8, 6, 5, 2, 0, 9, 5, 1])


In [ ]:
XX = X.reshape(3,3)

In [ ]:
YY = Y.reshape(3,3)

In [ ]:
vocab_size

10

In [ ]:
class NextWordRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x) # [3, 3, 100]
        out, hx = self.rnn(x) # [3, 3, 16], [1, 3, 16]
        out = out[:, -1, :] # [3, 16]
        out = self.fc(out) # [3, 10]
        return out




In [ ]:
model = NextWordRNN(10)

In [ ]:
outputs = model(XX)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
for epoch in range(300):

    optimizer.zero_grad()

    inputs = X.unsqueeze(1)
    outputs = model(inputs)

    loss = loss_fn(outputs, Y)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())


Epoch: 0 Loss: 2.680279493331909
Epoch: 50 Loss: 0.18796661496162415
Epoch: 100 Loss: 0.16796457767486572
Epoch: 150 Loss: 0.1625269055366516
Epoch: 200 Loss: 0.15987415611743927
Epoch: 250 Loss: 0.1583499312400818


In [ ]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out).item()

    return idx2word[pred]

In [ ]:

print(predict_next("is"))
print(predict_next("word2vec"))

fun
is
